<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Impedimentos Sociais, Ambientais e Climáticos</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Thales Sehn Körting, Gilberto Queiroz, Karine Ferreira, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 11 de abril de 2026
</div>


<br/>

</div>

Esse notebook aborda de forma simplificada as restrições legais para o acesso ao crédito rural com base em critérios sociais, ambientais e climáticos.

# <span style="color:#336699">Contextualização</span>
<hr style="border:1px solid #0077b9;">

Com as recentes atualizações do Conselho Monetário Nacional (como as Resoluções CMN Nº 5.193/2024, 5.267/2025 e 5.268/2025), a verificação de financiamentos tornou-se mais rigorosa. Hoje, exige-se a identificação da propriedade via Cadastro Ambiental Rural (CAR) e o monitoramento obrigatório por sensoriamento remoto para áreas superiores a 300 hectares, garantindo que os recursos financiados respeitem a conformidade socioambiental.

Na atividade prática de hoje, vamos analisar os impedimentos definidos para esses empreendimentos.

# <span style="color:#336699">5 - Áreas Quilombolas</span>
<hr style="border:1px solid #0077b9;">

A palavra Quilombo cujo significado provável está relacionado com acampamento guerreiro na floresta e pode ter sua etimologia da palavra derivada do Quimbundo (Kilombo) significando ‘acampamento’, ‘arraial’, ‘povoação’, ‘capital’, ‘união’ e ainda ‘exército’. Essa expressão e palavra amplamente utilizada em diversas circunstâncias da história do Brasil, foi primeiramente popularizada pela administração colonial, em suas leis, relatórios, atos e decretos para se referir às unidades de apoio mútuo criadas pelos rebeldes ao sistema escravista, bem como às suas lutas pelo fim da escravidão no país. Em seguida, foi também expressão dos afrodescendentes para designar a sua trajetória, conquista e liberdade, em amplas dimensões e significados.

De acordo com a Resolução CMN Nº 5.193/2024:

> 8 - Para fins de cumprimento ao disposto no art. 68 do Ato das Disposições Constitucionais Transitórias e no
Decreto nº 4.887, de 20 de novembro de 2003, não será concedido crédito rural a empreendimento situado em
imóvel rural total ou parcialmente inserido em terras tituladas, ou com título parcial, por remanescentes das
comunidades de quilombos. (Res CMN 5.268 art 1º)

> 9 - O item 8 não se aplica aos casos em que o proponente pertença ao grupo remanescente da comunidade do
quilombo na qual se situa a área do empreendimento. (Res CMN 5.193 art 1º)

Uma representação ilustrativa em forma de diagrama da restrição pode ser observada na Figura 5:

![car](https://i.imgur.com/TdCuAie.png "Terras Indígenas")

Prática: vamos simular a verificação de sobreposição.




**Configuração e Exemplo que NÂO atende a norma**

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import requests, zipfile, io
import geopandas as gpd

In [ ]:
empreendimento = {
    "geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
    "comunidade": "Quilombo dos Palmares"
}

In [ ]:
lista_quilombolas = [
    {"geometria": Polygon([(0.5, 0.5), (1.5, 0.5), (1.5, 1.5), (0.5, 1.5)]), "nome_comunidade": "Quilombo B"},
    {"geometria": Polygon([(5, 5), (6, 5), (6, 6), (5, 6)]), "nome_comunidade": "Quilombo Kalunga"}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, quilombo in enumerate(lista_quilombolas):
    x_q, y_q = quilombo["geometria"].exterior.xy

    label = 'Área Quilombola' if i == 0 else None

    ax.plot(x_q, y_q, color='#2ca02c', linewidth=2, label=label)
    ax.fill(x_q, y_q, color='#2ca02c', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Áreas Quilombolas')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, terra in enumerate(lista_quilombolas):

    # Verifica se a geometria do empreendimento possui sobreposição com a terra quilombola atual.
    if empreendimento["geometria"].intersects(terra["geometria"]):

        # Verifica se o proponente pertence à mesma comunidade quilombola da terra sobreposta.
        if empreendimento["comunidade"] == terra["nome_comunidade"]:
            # Se for o mesmo quilombo, a operação é permitida.
            pass

        # Caso o proponente não pertença àquela comunidade quilombola específica.
        else:
            imovel_atende_norma = False
            motivo_impedimento = f"proponente não pertence à comunidade quilombola local ({terra['nome_comunidade']})."
            break

    # Caso não exista sobreposição com esta terra quilombola.
    else:
        pass

# Verifica se o estado da aprovação permaneceu verdadeiro após a análise da lista.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
# Caso tenha sido encontrado um impedimento social.
else:
    print(f"O empreendimento não atende a norma porque houve: {motivo_impedimento}")

**Configuração e Exemplo atende a norma**

In [ ]:
empreendimento = {
    "geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
    "comunidade": "Quilombo dos Palmares"
}

In [ ]:
lista_quilombolas = [
    {"geometria": Polygon([(0.5, 0.5), (1.5, 0.5), (1.5, 1.5), (0.5, 1.5)]), "nome_comunidade": "Quilombo dos Palmares"},
    {"geometria": Polygon([(5, 5), (6, 5), (6, 6), (5, 6)]), "nome_comunidade": "Quilombo Kalunga"},
    {"geometria": Polygon([(10, 10), (11, 10), (11, 11), (10, 11)]), "nome_comunidade": "Quilombo Rio dos Macacos"}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, quilombo in enumerate(lista_quilombolas):
    x_q, y_q = quilombo["geometria"].exterior.xy

    label = 'Área Quilombola' if i == 0 else None

    ax.plot(x_q, y_q, color='#2ca02c', linewidth=2, label=label)
    ax.fill(x_q, y_q, color='#2ca02c', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Áreas Quilombolas')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, terra in enumerate(lista_quilombolas):

    # Verifica se a geometria do empreendimento possui sobreposição com a terra quilombola atual.
    if empreendimento["geometria"].intersects(terra["geometria"]):

        # Verifica se o proponente pertence à mesma comunidade quilombola da terra sobreposta.
        if empreendimento["comunidade"] == terra["nome_comunidade"]:
            # Se for o mesmo quilombo, a operação é permitida.
            pass

        # Caso o proponente não pertença àquela comunidade quilombola específica.
        else:
            imovel_atende_norma = False
            motivo_impedimento = f"proponente não pertence à comunidade quilombola local ({terra['nome_comunidade']})."
            break

    # Caso não exista sobreposição com esta terra quilombola.
    else:
        pass

# Verifica se o estado da aprovação permaneceu verdadeiro após a análise da lista.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
# Caso tenha sido encontrado um impedimento social.
else:
    print(f"O empreendimento não atende a norma porque houve: {motivo_impedimento}")

**Exemplo com dados reais**

**Configuração e Exemplo Não atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/mini_quilombos_sab.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./mini_quilombos_sab.zip")
file_path = "./mini_quilombos_sab.zip/mini_quilombos_sab.shp"

mini_quilombos_sab = gpd.read_file(file_path)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_5.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_5.zip")
file_path = "./poligono_5.zip/poligono_5.shp"

poligono_5 = gpd.read_file(file_path)

In [ ]:
mini_quilombos_sab

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_quilombos_sab.plot(ax=ax, color='#d62728', alpha=0.3)
mini_quilombos_sab.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_5.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_5.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Quilombos')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Quilombos')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
poligono_5 = poligono_5.to_crs(mini_quilombos_sab.crs)
geom_empreendimento = poligono_5.geometry.iloc[0]

mini_quilombos_sab_plano = mini_quilombos_sab.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_5.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_quilombos_sab.intersects(geom_empreendimento)
areas = mini_quilombos_sab_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área da Quilombo {i}! Área da sobreposição: {areas[i]:.2f} m².")


**Configuração e Exemplo que atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_6.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_6.zip")
file_path = "./poligono_6.zip/poligono_6.shp"

poligono_6 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_quilombos_sab.plot(ax=ax, color='#d62728', alpha=0.3)
mini_quilombos_sab.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_6.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_6.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Quilombos')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Quilombos')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
poligono_6 = poligono_6.to_crs(mini_quilombos_sab.crs)
geom_empreendimento = poligono_6.geometry.iloc[0]

mini_quilombos_sab_plano = mini_quilombos_sab.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_6.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_quilombos_sab.intersects(geom_empreendimento)
areas = mini_quilombos_sab_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área da Quilombo {i}! Área da sobreposição: {areas[i]:.2f} m².")
